In [1]:
import torch
import csv
from transformers import pipeline, BitsAndBytesConfig
from ipywidgets import FileUpload
from IPython.display import display
import os

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16
)

pipe = pipeline(
    "text-generation",
    model="meta-llama/Llama-3.2-1B", 
    device_map="auto",
    model_kwargs={"quantization_config": quantization_config}
)

Device set to use cuda:0


In [2]:
filePath = "/home/james/NLP-Social-Trends/data/mock_data.csv"

rows = []
with open(filePath, newline='') as csvfile:
    reader = csv.DictReader(csvfile)
    for row in reader:
        rows.append(row)


In [3]:
prompt = f"""Task: Extract information from each social media post.

For each post, provide:
- Post Number (sequential, starting at 1)
- Sentiment (Positive, Negative, or Neutral)
- Location (country)
- Platform
- Likes count
- Retweets count

Output format (one line per post):
Post Number: X, Sentiment: X, Location: X, Platform: X, Likes: X, Retweets: X

Dataset:
{rows}

Output:"""

In [4]:
response = pipe(prompt, max_new_tokens=200, temperature=0.3, do_sample=False, return_full_text=False)
output = response[0]["generated_text"]
print(output)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


 
Post Number: 1, Sentiment: Positive, Location: USA, Platform: Twitter, Likes: 120, Retweets: 500
Post Number: 2, Sentiment: Negative, Location: UK, Platform: Facebook, Likes: 5, Retweets: 2
Post Number: 3, Sentiment: Neutral, Location: Canada, Platform: Instagram, Likes: 45, Retweets: 0
Post Number: 4, Sentiment: Neutral, Location: Australia, Platform: Twitter, Likes: 12, Retweets: 5
Post Number: 5, Sentiment: Negative, Location: USA, Platform: Twitter, Likes: 500, Retweets: 120



In [5]:
with open("model_output.txt", "w") as f:
    f.write(output)